# MAE 6246 - Homework 8: Furuta Pendulum Estimation and Tracking
**Name:** _Enter your name._  
**Due:** Friday, December 4, 2026, at 9:00 AM (Eastern Time)  
**Total:** 100 points. **Late penalty:** 1 point per 10 minutes late.

Follow the Week 13 ball-and-beam sequence: **Kalman filtering → LQG stabilization → time-varying reference tracking**. Use the Furuta model from Homework 7. Complete the marked code and response cells; keep all plots, tables, brief derivations, and one embedded animation in this notebook.

Submit **one completed `.ipynb` file** through Blackboard. There is no separate handwritten part or video submission. Write derivations in the Markdown response cells; a clearly legible image of handwritten derivations may be inserted instead. The course policy on generative AI applies; include the required supplementary record when AI materially influences your work.

Keep this notebook, `mjsim.py`, and `hw08_support.py` together. In Colab, upload the two Python files from the bundle into the Files panel before running the notebook. Do not substitute an older wrapper: this version supports sampled matrices, noise covariance, and measurements beginning at $k=1$.

## Setup and fixed model
The state is

$$
x=[\phi,\theta,\dot\phi,\dot\theta]^T,
$$

where $\phi$ is the arm angle and $\theta=0$ is the upright pendulum. **This state order differs from the ball-and-beam notebook.** The input is arm torque. Internally use radians, radians/s, and N m.

Load all plant parameters and matrices from `mjsim.py`. Use $H$ (loaded as `dyn.C`) for the sensor matrix and $z$ for the noisy measurements of **both angles**. The angular rates are not measured. The tracked output will later be arm angle, selected by $C$ (loaded as `dyn.tracking_C`).

The supplied model fixes the control sample period at 0.002 s, physics step at 0.001 s, and applied-torque limit at 0.20 N m. Keep these and the physical model unchanged. The evaluator uses unwrapped angles; do not wrap angles, reset within a trial, or add swing-up, gain scheduling, or extra sensors.

Actual sensor standard deviations are $0.20^\circ$ for arm angle and $0.10^\circ$ for pendulum angle. The random seed is fixed at 6248. No random process disturbance is injected; the filter uses a small assumed process intensity to represent model uncertainty. The nonlinear experiments are performance tests, not an exact MMSE-optimality test.

In [ ]:
import importlib.util
import subprocess
import sys
from pathlib import Path

packages = {"mujoco": "mujoco==3.17.0", "numpy": "numpy", "scipy": "scipy",
            "matplotlib": "matplotlib", "imageio_ffmpeg": "imageio-ffmpeg"}
missing = [package for module, package in packages.items()
           if importlib.util.find_spec(module) is None]
if missing:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing])
for filename in ["mjsim.py", "hw08_support.py"]:
    if not Path(filename).is_file():
        raise FileNotFoundError(f"Upload {filename} from the Homework 8 bundle into this notebook's folder.")

%matplotlib inline
import numpy as np
import matplotlib.pyplot as plt
from scipy.linalg import solve_discrete_are
from IPython.display import display, Video
import mjsim
import hw08_support as hw

np.set_printoptions(precision=6, suppress=True)
dyn = hw.load_plant()
A, B, H, D = dyn.A, dyn.B, dyn.C, dyn.D
Ad, Bd = dyn.discrete()
x_eq, u_eq, z_eq = dyn.x_eq, dyn.u_eq, dyn.y_eq
n, m, p = dyn.n, dyn.m, dyn.p
C = dyn.tracking_C
W = dyn.noise_covariance(hw.QE)
R = hw.R_SENSOR.copy()
Qc, Rc = hw.QC.copy(), hw.RC.copy()
dyn.info()
print("Sensor H =\n", H)
print("Tracked-output C =", C)

## Part 1 - Kalman filtering (30 points)
### 1(a) Observability (5 points)
Construct the sampled observability matrix $\mathcal O=[H^T,(HA_d)^T,\ldots,(HA_d^{n-1})^T]^T$ and report its rank. In one sentence, explain whether the two unmeasured angular rates can be estimated.

In [ ]:
# YOUR CODE: construct the sampled observability matrix.
O = None
if O is None:
    raise NotImplementedError("Construct the observability matrix.")
print("Observability matrix =\n", O)
print("Rank =", np.linalg.matrix_rank(O))

**Response 1(a):** _Report the rank and explain what it implies._

### 1(b) From the covariance recursion to the algebraic equation (10 points)
Start from

$$
P_k^+=(I-K_kH)P_k^-,\qquad
P_{k+1}^-=A_dP_k^+A_d^T+W,
$$

$$
K_k=P_k^-H^T(HP_k^-H^T+R)^{-1}.
$$

Substitute to obtain one recursion for $P_k^-$. Then set $P_{k+1}^-=P_k^-=\Pi$ and write the steady-state equation. State whether $\Pi$ is a prior or posterior covariance and give the formula for the fixed correction gain $K_\infty$.

**Response 1(b):** _Show the substitution and the resulting equations._

### 1(c) Fixed-gain estimator (15 points)
Implement the steady-state gain calculation and the observer callback below. Use the supplied $W,R$; report $K_\infty$ and the diagonal of $\Pi$. The SciPy command is `solve_discrete_are(Ad.T, H.T, W, R)`.

There is **no measurement at $k=0$**. Initialize $\hat x_0=x_{\mathrm{eq}}$, apply $u_0$, and predict to $k=1$. The observer receives `z=None` at $k=0$ and must skip correction. For $k\geq1$, the callback receives a prior, corrects using $z_k$, and returns the next prior:

$$
\nu_k=z_k-z_{\mathrm{eq}}-H(\hat x_k^--x_{\mathrm{eq}})-D(u_k-u_{\mathrm{eq}}),
$$

$$
\hat x_k^+=\hat x_k^-+K_\infty\nu_k,
$$

$$
\hat x_{k+1}^-=x_{\mathrm{eq}}+A_d(\hat x_k^+-x_{\mathrm{eq}})+B_d(u_k-u_{\mathrm{eq}}).
$$

The wrapper supplies the saturated applied input. The gain is fixed; no online covariance iteration is required.

In [ ]:
def design_kalman(Ad, H, W, R_assumed):
    # YOUR CODE: steady prior covariance and fixed correction gain.
    Pi = None
    K_inf = None
    if Pi is None or K_inf is None:
        raise NotImplementedError("Compute Pi and K_inf.")
    return Pi, K_inf

Pi, K_inf = design_kalman(Ad, H, W, R)
print("Diagonal of Pi =", np.diag(Pi))
print("Fixed gain K_inf =\n", K_inf)

In [ ]:
def make_observer(dyn, K_inf):
    Ad, Bd = dyn.discrete()
    H, D = dyn.C, dyn.D
    def observer(t, xhat_prior, u_applied, z):
        # YOUR CODE: correct when z is available, then predict to k+1.
        # Return the next prior in physical coordinates.
        raise NotImplementedError("Implement correction and prediction, including z=None.")
    return observer

## Part 2 - LQG stabilization (30 points)
### 2(a) Infinite-horizon LQR and state feedback (8 points)
Use the supplied discrete LQR weights

$$
Q_c=\operatorname{diag}(1,10,0.1,0.1),\qquad R_c=[100].
$$

Compute the infinite-horizon Riccati solution $X$ and gain $F$ for $(A_d,B_d)$, and implement `make_controller(dyn, F)` using $u=u_{\mathrm{eq}}-F(\hat x-x_{\mathrm{eq}})$. The Riccati solver returns $X$, not $F$. Report $F$; use this same gain in Parts 2 and 3.

In [ ]:
# YOUR CODE: discrete LQR Riccati solution and feedback gain.
X = None
F = None
if X is None or F is None:
    raise NotImplementedError("Compute X and F for the supplied weights.")
print("LQR gain F =", F)


def make_controller(dyn, F):
    def controller(t, xhat):
        # YOUR CODE: return requested torque, in physical coordinates.
        raise NotImplementedError("Implement state feedback.")
    return controller

### 2(b) Stabilization with estimated states (17 points)
Run the supplied full-state LQR baseline and LQG experiment. Both start from $x_0=[10^\circ,5^\circ,0,0]^T$ and last 8 s. The LQG initial estimate is zero. Include the comparison plot and table.

The LQG controller must meet these **three requirements**:

| Quantity | Actual constraint |
|---|---|
| Applied torque for the full trial | $\lvert u\rvert\leq0.20$ N m |
| Arm angle at every sample from 4 to 8 s | $\lvert \phi\rvert\leq2^\circ$ |
| Pendulum angle at every sample from 4 to 8 s | $\lvert \theta\rvert\leq1^\circ$ |

In a short paragraph, compare the initial transient and residual motion with full-state feedback. The LQG controller uses the prior $\hat x_k^-$; correction from $z_k$ affects the prediction used at $k+1$. True state is available only to the baseline and to evaluation plots.

In [ ]:
full_state = hw.run(dyn, make_controller(dyn, F))
lqg = hw.run(dyn, make_controller(dyn, F), make_observer(dyn, K_inf))
records = {"Full-state LQR": full_state, "LQG": lqg}
stabilization_metrics = hw.results_table(records)
fig = mjsim.plot_response(list(records.values()), labels=list(records),
                         title="Furuta stabilization: true states and prior estimates")
display(fig)
plt.close(fig)
assert stabilization_metrics["LQG"]["passed"], "Review the controller, gain, and observer timing."

**Response 2(b):** _Compare the two responses and identify the effect of estimation._

### 2(c) Assumed noise versus actual noise (5 points)
Repeat only the LQG stabilization test with the filter assuming $R_{\mathrm{assumed}}=9R$. Keep the **actual sensor noise**, process intensity, LQR gain, initial condition, and seed fixed. Use the provided table of angular-rate estimation RMSE over 4–8 s.

Explain in two or three sentences what changes in the gain and estimation response. Do not assume that one realization must improve monotonically. The Part 2(b) pass requirements apply to the baseline filter; this comparison has no additional pass threshold.

In [ ]:
Pi_9R, K_9R = design_kalman(Ad, H, W, 9*R)
print("Gain with assumed 9R =\n", K_9R)
lqg_9R = hw.run(dyn, make_controller(dyn, F), make_observer(dyn, K_9R))
hw.estimation_table({"Assumed R": lqg, "Assumed 9R": lqg_9R})

**Response 2(c):** _Interpret the comparison while distinguishing assumed from actual noise._

## Part 3 - Reference tracking (40 points)
### 3(a) Construct a feasible sinusoidal reference (15 points)
Track arm angle

$$
r(t)=a\sin(\omega t),\qquad a=30^\circ\ \text{in radians},\qquad\omega=2\pi/6\ \mathrm{rad/s}.
$$

The reference must satisfy $\dot x_r=Ax_r+Bu_r$ and $Cx_r=r(t)$. The pendulum generally must lean slightly during accelerated arm motion; do not impose $\theta_r=0$ for the moving trajectory.

Use the harmonic form

$$
x_r(t)=\begin{bmatrix}
a\sin(\omega t) \\
b\sin(\omega t) \\
a\omega\cos(\omega t) \\
b\omega\cos(\omega t)
\end{bmatrix},\qquad u_r(t)=c\sin(\omega t).
$$

Here $b$ is an angular amplitude and $c$ is a torque amplitude; both may be negative. The first two state equations are automatically satisfied. Substitute into the **last two rows of the loaded continuous matrix equation** to obtain two linear equations for $b,c$. Write the resulting $2\times2$ system, solve it, and implement `sinusoidal_reference(t)`.

Use the loaded entries of $A,B$; do not copy numerical matrices into the notebook. The continuous reference is sampled during simulation, so small discretization and nonlinear-model errors remain.

**Response 3(a):** _Derive the two scalar equations and the 2-by-2 system for b and c._

In [ ]:
a, omega = hw.REFERENCE_AMPLITUDE, hw.OMEGA
# YOUR CODE: construct the two equations for the harmonic amplitudes.
harmonic_matrix = None
harmonic_rhs = None
if harmonic_matrix is None or harmonic_rhs is None:
    raise NotImplementedError("Construct the 2-by-2 harmonic reference system.")
b, c = np.linalg.solve(harmonic_matrix, harmonic_rhs)
print("Pendulum reference amplitude b [deg] =", np.rad2deg(b))
print("Torque reference amplitude c [N m] =", c)


def sinusoidal_reference(t):
    # YOUR CODE: return xr (shape n,) and ur (shape m,) in physical coordinates.
    raise NotImplementedError("Implement the feasible sinusoidal reference.")

In [ ]:
# Supplied check of continuous reference feasibility; not an extra subproblem.
for tk in np.linspace(0.0, hw.T_SINUSOID, 61):
    xr, ur = sinusoidal_reference(tk)
    eps = 1e-5
    xr_dot = (sinusoidal_reference(tk+eps)[0] - sinusoidal_reference(tk-eps)[0])/(2*eps)
    assert np.allclose(A@(xr-x_eq)+B@(ur-u_eq), xr_dot, atol=1e-7)
    assert np.allclose(C@(xr-x_eq), a*np.sin(omega*tk))
print("Reference equations verified.")

### 3(b) Stabilize the new state (20 points)
Introduce $\tilde x=x-x_r$ and $\tilde u=u-u_r$. Show that $\dot{\tilde x}=A\tilde x+B\tilde u$, so $\tilde x=0$ corresponds to perfect tracking. Extend `make_controller` to use

$$
u=u_r-F(\hat x-x_r).
$$

Use the **same $F$ and baseline $K_\infty$** as in Part 2. The observer estimates the actual state; reference subtraction belongs in the controller. The supplied simulation starts from the upright equilibrium with zero estimate and runs for three periods (18 s). The initial reference velocity is nonzero, so an initial tracking transient is expected.

Include the supplied tracking-error plots and results table. Requirements:

| Quantity | Actual constraint |
|---|---|
| Applied torque for the full trial | $\lvert u\rvert\leq0.20$ N m |
| Arm tracking RMSE over 6–18 s | $\sqrt{\operatorname{mean}[(\phi-r)^2]}\leq1^\circ$ |
| Pendulum angle at every sample over 0–18 s | $\lvert \theta\rvert\leq5^\circ$ |

The true pendulum angle need not be zero during the sinusoid; its reference component is $b\sin(\omega t)$. The first four plot panels show $x-x_r$.

**Response 3(b):** _Show the new-state dynamics and explain why stabilization of its origin gives tracking._

In [ ]:
def make_controller(dyn, F, reference=None):
    def controller(t, xhat):
        # YOUR CODE: use the equilibrium if reference is None;
        # otherwise obtain xr, ur from reference(t), then use xhat - xr.
        raise NotImplementedError("Extend the controller to reference-state feedback.")
    return controller

In [ ]:
sinusoid = hw.run(dyn, make_controller(dyn, F, sinusoidal_reference),
                   make_observer(dyn, K_inf), experiment="sinusoid")
sinusoid_metrics = hw.results_table({"LQG tracking": sinusoid}, experiment="sinusoid")
fig = hw.plot_tracking(sinusoid, sinusoidal_reference, "Sinusoidal tracking: x - xr")
display(fig)
plt.close(fig)
assert sinusoid_metrics["LQG tracking"]["passed"], "Review the reference and tracking controller."

### 3(c) Animation (5 points)
Run the cell below to embed the nonlinear **sinusoidal-tracking** animation. Save and reopen the notebook to confirm the video remains embedded. No additional simulation or separate video submission is needed.

In [ ]:
video_path = mjsim.animate(sinusoid, "homework08_furuta_tracking.mp4", fps=30)
display(Video(video_path, embed=True))

## Submission checklist
- [ ] I completed the brief derivations and all marked code and response cells.
- [ ] I used H and z for the sensor model and loaded plant matrices from mjsim.py.
- [ ] My observer skips correction at k=0 and uses a fixed gain from the steady prior covariance.
- [ ] I included the full-state/LQG comparison and the assumed-noise comparison.
- [ ] My baseline LQG and the sinusoidal tracking experiment meet the displayed requirements.
- [ ] I used the same F and baseline Kalman gain for stabilization and tracking.
- [ ] The saved notebook contains the sinusoidal-tracking animation.
- [ ] I restarted the kernel, ran all cells, saved, and reopened the notebook.

Use the Week 13 classroom notebook and lecture notes as your primary references. The assignment is graded on the derivations, implementation, interpretation, and requested evidence; meeting numerical limits alone does not earn full credit.